In [8]:
# Import 라이브러리
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import psycopg2
import json
from datetime import datetime
import logging
from tqdm import tqdm
import os
import math
import copy
from sklearn.model_selection import train_test_split

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("=" * 70)
print("🚀 Beta-VAE 고급 통합 임베딩 파이프라인 시작")
print("🔥 GPU 가속 | Early Stopping | Warm-up | LR Scheduler | Mixed Precision")
print("=" * 70)
logger.info("필요한 라이브러리가 성공적으로 로드되었습니다")

# GPU 설정 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 사용 디바이스: {device}")
if torch.cuda.is_available():
    print(f"   🔸 GPU: {torch.cuda.get_device_name()}")
    print(f"   🔸 GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    try:
        import torch.version
        print(f"   🔸 CUDA 버전: {torch.version.cuda}")
    except:
        print(f"   🔸 PyTorch 버전: {torch.__version__}")
else:
    print("   ⚠️ CUDA를 사용할 수 없습니다. CPU로 훈련합니다.")

logger.info(f"디바이스 설정 완료: {device}")


2025-06-23 02:09:05,930 - INFO - 필요한 라이브러리가 성공적으로 로드되었습니다
2025-06-23 02:09:05,931 - INFO - 디바이스 설정 완료: cuda


🚀 Beta-VAE 고급 통합 임베딩 파이프라인 시작
🔥 GPU 가속 | Early Stopping | Warm-up | LR Scheduler | Mixed Precision
🎯 사용 디바이스: cuda
   🔸 GPU: NVIDIA GeForce GTX 1080 Ti
   🔸 GPU 메모리: 11.0 GB
   🔸 CUDA 버전: 12.1


In [9]:
# DB 연결
print("\n📊 1단계: 데이터베이스 연결")
print("-" * 40)

try:
    logger.info("PostgreSQL 데이터베이스 연결 시도 중...")
    conn = psycopg2.connect(
        dbname='postgres', user='postgres', password='postgres', host='localhost', port=5432
    )
    cursor = conn.cursor()
    logger.info("✅ 데이터베이스 연결 성공")
    print("✅ 데이터베이스 연결 완료")
except Exception as e:
    logger.error(f"❌ 데이터베이스 연결 실패: {e}")
    raise


2025-06-23 02:09:05,953 - INFO - PostgreSQL 데이터베이스 연결 시도 중...
2025-06-23 02:09:06,071 - INFO - ✅ 데이터베이스 연결 성공



📊 1단계: 데이터베이스 연결
----------------------------------------
✅ 데이터베이스 연결 완료


In [10]:
# 데이터 로드 (wavelet + dct)
print("\n📊 2단계: 데이터 로딩")
print("-" * 40)

def load_vectors(table_name: str, dim_limit: int = 512):
    logger.info(f"테이블 '{table_name}'에서 벡터 로딩 시작 (차원 제한: {dim_limit})")
    query = f"""
        SELECT embedding FROM {table_name}
        WHERE embedding IS NOT NULL AND compressed_dim <= {dim_limit}
          AND compression_ratio <= 1.0
    """
    cursor.execute(query)
    rows = cursor.fetchall()
    logger.info(f"쿼리 실행 완료: {len(rows)}개 레코드 발견")
    
    print(f"   📁 {table_name} 테이블에서 벡터 파싱 중...")
    vectors = []
    for i, row in enumerate(tqdm(rows, desc=f"   {table_name} 파싱", leave=False)):
        try:
            vectors.append(np.array(json.loads(row[0])))
        except Exception as e:
            logger.warning(f"벡터 파싱 실패 (행 {i}): {e}")
    
    logger.info(f"✅ '{table_name}' 로딩 완료: {len(vectors)}개 벡터")
    return vectors

print("📥 Wavelet 벡터 로딩 중...")
wavelet_vectors = load_vectors('wavelet_vector')

print("📥 DCT 벡터 로딩 중...")
dct_vectors = load_vectors('dct_vector')

print("\n📊 데이터 통합 및 분석 중...")
all_vectors = wavelet_vectors + dct_vectors
original_dims = [len(vec) for vec in all_vectors]
max_dim = max(original_dims) if original_dims else 0

print("\n📈 데이터 요약:")
print(f"   🔸 Wavelet 벡터: {len(wavelet_vectors):,}개")
print(f"   🔸 DCT 벡터: {len(dct_vectors):,}개")
print(f"   🔸 총 벡터 수: {len(all_vectors):,}개")
print(f"   🔸 최대 차원: {max_dim}")

logger.info(f"데이터 로딩 완료: 총 {len(all_vectors)}개 벡터, 최대 차원 {max_dim}")

2025-06-23 02:09:06,097 - INFO - 테이블 'wavelet_vector'에서 벡터 로딩 시작 (차원 제한: 512)



📊 2단계: 데이터 로딩
----------------------------------------
📥 Wavelet 벡터 로딩 중...


2025-06-23 02:09:27,943 - INFO - 쿼리 실행 완료: 1213940개 레코드 발견


   📁 wavelet_vector 테이블에서 벡터 파싱 중...


2025-06-23 02:10:17,668 - INFO - ✅ 'wavelet_vector' 로딩 완료: 1213940개 벡터              
2025-06-23 02:10:18,409 - INFO - 테이블 'dct_vector'에서 벡터 로딩 시작 (차원 제한: 512)


📥 DCT 벡터 로딩 중...


2025-06-23 02:10:22,014 - INFO - 쿼리 실행 완료: 158340개 레코드 발견


   📁 dct_vector 테이블에서 벡터 파싱 중...


2025-06-23 02:10:29,303 - INFO - ✅ 'dct_vector' 로딩 완료: 158340개 벡터             
2025-06-23 02:10:30,282 - INFO - 데이터 로딩 완료: 총 1372280개 벡터, 최대 차원 511



📊 데이터 통합 및 분석 중...

📈 데이터 요약:
   🔸 Wavelet 벡터: 1,213,940개
   🔸 DCT 벡터: 158,340개
   🔸 총 벡터 수: 1,372,280개
   🔸 최대 차원: 511


In [11]:
# 벡터 패딩
print("\n📊 3단계: 벡터 패딩 및 텐서 변환")
print("-" * 40)

logger.info(f"벡터 패딩 시작: 목표 차원 {max_dim}")
print(f"📐 모든 벡터를 {max_dim} 차원으로 패딩 중...")

padded_vectors = []
for i, vec in enumerate(tqdm(all_vectors, desc="   벡터 패딩", leave=False)):
    try:
        padded_vec = np.pad(vec, (0, max_dim - len(vec)))
        padded_vectors.append(padded_vec)
    except Exception as e:
        logger.error(f"벡터 패딩 실패 (인덱스 {i}): {e}")
        raise

logger.info("패딩 완료, PyTorch 텐서로 변환 중...")
X = torch.tensor(padded_vectors, dtype=torch.float32)  # CPU에 유지

print("\n📊 텐서 정보:")
print(f"   🔸 패딩된 벡터 모양: {X.shape}")
print(f"   🔸 데이터 타입: {X.dtype}")
print(f"   🔸 메모리 사용량: {X.element_size() * X.nelement() / 1024**2:.2f} MB")

# 훈련/검증 데이터 분할
print(f"\n🔄 훈련/검증 데이터 분할 중...")
logger.info("훈련/검증 데이터 분할 시작")

# 인덱스 기반으로 분할 (메모리 효율성을 위해)
n_samples = len(X)
indices = np.arange(n_samples)
train_indices, val_indices = train_test_split(
    indices, test_size=0.15, random_state=42, shuffle=True
)

print(f"   🔸 전체 데이터: {n_samples:,}개")
print(f"   🔸 훈련 데이터: {len(train_indices):,}개 ({len(train_indices)/n_samples*100:.1f}%)")
print(f"   🔸 검증 데이터: {len(val_indices):,}개 ({len(val_indices)/n_samples*100:.1f}%)")

logger.info(f"벡터 패딩 및 텐서 변환 완료: {X.shape}")
logger.info(f"데이터 분할 완료: 훈련 {len(train_indices):,}개, 검증 {len(val_indices):,}개")

# 메모리 정리
del padded_vectors
import gc
gc.collect()
logger.info("임시 메모리 정리 완료")


2025-06-23 02:10:30,298 - INFO - 벡터 패딩 시작: 목표 차원 511



📊 3단계: 벡터 패딩 및 텐서 변환
----------------------------------------
📐 모든 벡터를 511 차원으로 패딩 중...


2025-06-23 02:10:54,407 - INFO - 패딩 완료, PyTorch 텐서로 변환 중...               
2025-06-23 02:11:54,495 - INFO - 훈련/검증 데이터 분할 시작
2025-06-23 02:11:54,565 - INFO - 벡터 패딩 및 텐서 변환 완료: torch.Size([1372280, 511])
2025-06-23 02:11:54,567 - INFO - 데이터 분할 완료: 훈련 1,166,438개, 검증 205,842개
2025-06-23 02:11:57,163 - INFO - 임시 메모리 정리 완료



📊 텐서 정보:
   🔸 패딩된 벡터 모양: torch.Size([1372280, 511])
   🔸 데이터 타입: torch.float32
   🔸 메모리 사용량: 2675.00 MB

🔄 훈련/검증 데이터 분할 중...
   🔸 전체 데이터: 1,372,280개
   🔸 훈련 데이터: 1,166,438개 (85.0%)
   🔸 검증 데이터: 205,842개 (15.0%)


In [12]:
# Beta-VAE 모델 정의 (고급 기법 적용)
print("\n📊 4단계: 고급 Beta-VAE 모델 정의")
print("-" * 40)

logger.info("고급 Beta-VAE 모델 클래스 정의 시작")

class AdvancedBetaVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=128, beta=4.0, dropout_rate=0.1):
        super(AdvancedBetaVAE, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.beta = beta
        
        # 인코더 (Layer Normalization + Dropout 적용)
        self.fc1 = nn.Linear(input_dim, 512)
        self.ln1 = nn.LayerNorm(512)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(512, 256)
        self.ln2 = nn.LayerNorm(256)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc21 = nn.Linear(256, latent_dim)  # mu
        self.fc22 = nn.Linear(256, latent_dim)  # logvar
        
        # 디코더 (Layer Normalization + Dropout 적용)
        self.fc3 = nn.Linear(latent_dim, 256)
        self.ln3 = nn.LayerNorm(256)
        self.dropout3 = nn.Dropout(dropout_rate)
        
        self.fc4 = nn.Linear(256, 512)
        self.ln4 = nn.LayerNorm(512)
        self.dropout4 = nn.Dropout(dropout_rate)
        
        self.fc5 = nn.Linear(512, input_dim)
        
        # 가중치 초기화
        self._initialize_weights()
        
        logger.info(f"고급 모델 구조: {input_dim} -> 512 -> 256 -> {latent_dim} -> 256 -> 512 -> {input_dim}")
        logger.info(f"Beta 값: {beta}, Dropout 비율: {dropout_rate}")

    def _initialize_weights(self):
        """Xavier/Glorot 초기화 적용"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def encode(self, x):
        h1 = self.dropout1(F.relu(self.ln1(self.fc1(x))))
        h2 = self.dropout2(F.relu(self.ln2(self.fc2(h1))))
        return self.fc21(h2), self.fc22(h2)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = self.dropout3(F.relu(self.ln3(self.fc3(z))))
        h4 = self.dropout4(F.relu(self.ln4(self.fc4(h3))))
        return self.fc5(h4)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def loss_function(self, recon_x, x, mu, logvar, mask, beta_weight=1.0):
        # 마스크 기반 재구성 손실
        recon_loss = ((recon_x - x) ** 2 * mask).sum() / mask.sum()
        
        # KL Divergence 손실
        kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
        
        # Beta 가중치 적용 (Warm-up 지원)
        total_loss = recon_loss + self.beta * beta_weight * kld
        
        return total_loss, recon_loss, kld

# 고급 체크포인트 관리자 클래스
class AdvancedCheckpointManager:
    def __init__(self, checkpoint_dir, experiment_id, save_best=True, save_every_n=None):
        self.checkpoint_dir = checkpoint_dir
        self.experiment_id = experiment_id
        self.save_best = save_best
        self.save_every_n = save_every_n
        self.best_loss = float('inf')
        self.best_epoch = 0
        
        # 체크포인트 파일 경로들
        self.best_checkpoint_path = os.path.join(checkpoint_dir, f"best_model_{experiment_id}.pth")
        self.latest_checkpoint_path = os.path.join(checkpoint_dir, f"latest_model_{experiment_id}.pth")
        self.config_path = os.path.join(checkpoint_dir, f"config_{experiment_id}.json")
        
    def save_checkpoint(self, epoch, model, optimizer, train_losses, val_losses, 
                       hyperparameters, is_best=False, additional_info=None):
        """포괄적인 체크포인트 저장"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val_loss': self.best_loss,
            'best_epoch': self.best_epoch,
            'experiment_id': self.experiment_id,
            'hyperparameters': hyperparameters,
            'timestamp': datetime.now().isoformat(),
        }
        
        if additional_info:
            checkpoint.update(additional_info)
        
        # 항상 최신 체크포인트 저장
        torch.save(checkpoint, self.latest_checkpoint_path)
        
        # 최적 모델 저장
        if is_best and self.save_best:
            torch.save(checkpoint, self.best_checkpoint_path)
            logger.info(f"💾 최적 모델 저장됨: {self.best_checkpoint_path}")
        
        # 주기적 체크포인트 저장
        if self.save_every_n and (epoch + 1) % self.save_every_n == 0:
            periodic_path = os.path.join(self.checkpoint_dir, f"epoch_{epoch+1:03d}_{self.experiment_id}.pth")
            torch.save(checkpoint, periodic_path)
            logger.info(f"📁 주기적 체크포인트 저장: {periodic_path}")
        
        return checkpoint
    
    def update_best(self, val_loss, epoch):
        """최적 성능 업데이트"""
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_epoch = epoch
            return True
        return False
    
    def save_config(self, config):
        """설정 파일 저장"""
        with open(self.config_path, 'w') as f:
            json.dump(config, f, indent=2, default=str)
        logger.info(f"⚙️ 설정 파일 저장: {self.config_path}")

# 향상된 Early Stopping 클래스
class AdvancedEarlyStopping:
    def __init__(self, patience=10, min_delta=1e-6, restore_best_weights=True, 
                 checkpoint_manager=None):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.checkpoint_manager = checkpoint_manager
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
        self.stopped_epoch = 0
        
    def __call__(self, val_loss, model, epoch):
        is_best = False
        
        if self.best_loss is None:
            self.best_loss = val_loss
            is_best = True
            self.save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            is_best = True
            self.save_checkpoint(model)
        else:
            self.counter += 1
        
        # 체크포인트 매니저에 최적 성능 업데이트
        if self.checkpoint_manager and is_best:
            self.checkpoint_manager.update_best(val_loss, epoch)
            
        if self.counter >= self.patience:
            self.stopped_epoch = epoch
            if self.restore_best_weights and self.best_weights is not None:
                model.load_state_dict(self.best_weights)
                logger.info(f"🔄 최적 가중치로 복원됨 (에포크 {epoch - self.counter})")
            return True
        return False
    
    def save_checkpoint(self, model):
        """최적 모델 가중치 저장"""
        self.best_weights = copy.deepcopy(model.state_dict())

# Learning Rate Warm-up Scheduler
class WarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, base_lr):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.base_lr = base_lr
        self.current_lr = 0
        
    def step(self, epoch):
        if epoch < self.warmup_epochs:
            # Linear warm-up
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
        else:
            lr = self.base_lr
            
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        self.current_lr = lr
        return lr

print("✅ 고급 Beta-VAE 모델 및 유틸리티 클래스 정의 완료")
print("   🔸 Layer Normalization + Dropout 정규화")
print("   🔸 Xavier 가중치 초기화")
print("   🔸 Early Stopping 지원")
print("   🔸 Warm-up Learning Rate Scheduler")
logger.info("고급 Beta-VAE 모델 정의 완료")


2025-06-23 02:11:57,197 - INFO - 고급 Beta-VAE 모델 클래스 정의 시작
2025-06-23 02:11:57,199 - INFO - 고급 Beta-VAE 모델 정의 완료



📊 4단계: 고급 Beta-VAE 모델 정의
----------------------------------------
✅ 고급 Beta-VAE 모델 및 유틸리티 클래스 정의 완료
   🔸 Layer Normalization + Dropout 정규화
   🔸 Xavier 가중치 초기화
   🔸 Early Stopping 지원
   🔸 Warm-up Learning Rate Scheduler


In [13]:
# 고급 훈련 설정
print("\n📊 5단계: 고급 모델 초기화 및 훈련 준비")
print("-" * 40)

# 체크포인트 디렉토리 설정
checkpoint_dir = "advanced_b_vae_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"📁 체크포인트 디렉토리 생성: {checkpoint_dir}")

# 실험 ID 생성 (타임스탬프 기반)
experiment_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"🆔 실험 ID: {experiment_id}")

# 고급 하이퍼파라미터 설정
input_dim = max_dim
latent_dim = 128
beta = 4.0
base_lr = 2e-4  # 더 안정적인 학습률
weight_decay = 1e-5  # 가중치 감쇠
epochs = 100
batch_size = 512  # GPU 메모리에 따라 조정
dropout_rate = 0.15
warmup_epochs = 10
patience = 15
grad_clip_value = 1.0
save_every_n_epochs = 10  # N 에포크마다 중간 체크포인트 저장

# GPU 메모리에 따른 배치 크기 자동 조정
if device.type == 'cuda':
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    if gpu_memory_gb < 8:
        batch_size = 256
    elif gpu_memory_gb < 16:
        batch_size = 512
    else:
        batch_size = 1024
    print(f"🎯 GPU 메모리 기반 배치 크기 조정: {batch_size}")

logger.info("고급 모델 인스턴스 생성 중...")
vae = AdvancedBetaVAE(
    input_dim=input_dim, 
    latent_dim=latent_dim, 
    beta=beta,
    dropout_rate=dropout_rate
).to(device)

# 파라미터 수 계산
total_params = sum(p.numel() for p in vae.parameters())
trainable_params = sum(p.numel() for p in vae.parameters() if p.requires_grad)

print(f"\n🧠 고급 모델 정보:")
print(f"   🔸 총 파라미터 수: {total_params:,}")
print(f"   🔸 훈련 가능 파라미터: {trainable_params:,}")
print(f"   🔸 모델 크기: {total_params * 4 / 1024**2:.2f} MB")

# 고급 옵티마이저 설정 (AdamW with weight decay)
logger.info("고급 옵티마이저 및 스케줄러 설정 중...")
optimizer = optim.AdamW(
    vae.parameters(), 
    lr=base_lr, 
    weight_decay=weight_decay,
    betas=(0.9, 0.999),
    eps=1e-8
)

# Learning Rate Schedulers
warmup_scheduler = WarmupScheduler(optimizer, warmup_epochs, base_lr)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs, eta_min=base_lr*0.01)
plateau_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=8)

# 고급 체크포인트 매니저 및 Early Stopping 설정
checkpoint_manager = AdvancedCheckpointManager(
    checkpoint_dir=checkpoint_dir,
    experiment_id=experiment_id,
    save_best=True,
    save_every_n=save_every_n_epochs
)

early_stopping = AdvancedEarlyStopping(
    patience=patience, 
    min_delta=1e-6,
    checkpoint_manager=checkpoint_manager
)

# 하이퍼파라미터 설정 저장
hyperparameters = {
    'input_dim': input_dim,
    'latent_dim': latent_dim,
    'beta': beta,
    'dropout_rate': dropout_rate,
    'base_lr': base_lr,
    'weight_decay': weight_decay,
    'batch_size': batch_size,
    'epochs': epochs,
    'warmup_epochs': warmup_epochs,
    'patience': patience,
    'grad_clip_value': grad_clip_value,
    'device': str(device),
    'total_params': total_params,
    'trainable_params': trainable_params
}
checkpoint_manager.save_config(hyperparameters)

# Mixed Precision Training
scaler = GradScaler() if device.type == 'cuda' else None

# 마스크 생성 (GPU 이동 전에 데이터로더 설정)
logger.info("마스크 생성 및 데이터로더 설정 중...")
print("🎭 패딩 마스크 생성 중...")
mask = torch.tensor([[1.0] * d + [0.0] * (max_dim - d) for d in original_dims])

# 데이터로더 설정 (GPU 이동 전에 CPU에서 설정)
logger.info("고급 데이터로더 설정 중...")

# 훈련 데이터셋 (CPU 텐서로)
train_dataset = data.TensorDataset(X[train_indices], mask[train_indices])
train_dataloader = data.DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True,
    num_workers=0,  # Windows 호환성
    pin_memory=True if device.type == 'cuda' else False  # CPU 텐서이므로 pin_memory 가능
)

# 검증 데이터셋 (CPU 텐서로)
val_dataset = data.TensorDataset(X[val_indices], mask[val_indices])
val_dataloader = data.DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False,
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False  # CPU 텐서이므로 pin_memory 가능
)

# 이제 GPU로 이동 (데이터로더에서 배치별로 이동됨)
logger.info("데이터로더 설정 완료, 배치는 훈련 중 GPU로 자동 이동됨")

print(f"\n⚙️ 고급 훈련 설정:")
print(f"   🔸 모델 입력 차원: {input_dim}")
print(f"   🔸 잠재 벡터 차원: {latent_dim}")
print(f"   🔸 Beta 값: {beta}")
print(f"   🔸 기본 학습률: {base_lr}")
print(f"   🔸 가중치 감쇠: {weight_decay}")
print(f"   🔸 Dropout 비율: {dropout_rate}")
print(f"   🔸 배치 크기: {batch_size}")
print(f"   🔸 에포크 수: {epochs}")
print(f"   🔸 Warm-up 에포크: {warmup_epochs}")
print(f"   🔸 Early Stopping 인내심: {patience}")
print(f"   🔸 Gradient Clipping: {grad_clip_value}")
print(f"   🔸 Mixed Precision: {'✅' if scaler else '❌'}")
print(f"   🔸 훈련 배치 수: {len(train_dataloader):,}")
print(f"   🔸 검증 배치 수: {len(val_dataloader):,}")

print(f"\n💾 체크포인트 설정:")
print(f"   🔸 체크포인트 디렉토리: {checkpoint_dir}")
print(f"   🔸 실험 ID: {experiment_id}")
print(f"   🔸 최적 모델 저장: ✅")
print(f"   🔸 주기적 저장: 매 {save_every_n_epochs} 에포크")
print(f"   🔸 최적 모델 경로: {checkpoint_manager.best_checkpoint_path}")
print(f"   🔸 최신 모델 경로: {checkpoint_manager.latest_checkpoint_path}")
print(f"   🔸 설정 파일 경로: {checkpoint_manager.config_path}")

logger.info("고급 모델 초기화 및 훈련 준비 완료")


2025-06-23 02:11:57,234 - INFO - 고급 모델 인스턴스 생성 중...
2025-06-23 02:11:57,243 - INFO - 고급 모델 구조: 511 -> 512 -> 256 -> 128 -> 256 -> 512 -> 511
2025-06-23 02:11:57,244 - INFO - Beta 값: 4.0, Dropout 비율: 0.15
2025-06-23 02:11:57,250 - INFO - 고급 옵티마이저 및 스케줄러 설정 중...
2025-06-23 02:11:57,252 - INFO - ⚙️ 설정 파일 저장: notebooks/advanced_b_vae_checkpoints\config_20250623_021157.json
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5592\139976545.py:105: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if device.type == 'cuda' else None
2025-06-23 02:11:57,254 - INFO - 마스크 생성 및 데이터로더 설정 중...



📊 5단계: 고급 모델 초기화 및 훈련 준비
----------------------------------------
📁 체크포인트 디렉토리 생성: notebooks/advanced_b_vae_checkpoints
🆔 실험 ID: 20250623_021157
🎯 GPU 메모리 기반 배치 크기 조정: 512

🧠 고급 모델 정보:
   🔸 총 파라미터 수: 889,087
   🔸 훈련 가능 파라미터: 889,087
   🔸 모델 크기: 3.39 MB
🎭 패딩 마스크 생성 중...


2025-06-23 02:13:16,668 - INFO - 고급 데이터로더 설정 중...
2025-06-23 02:13:17,383 - INFO - 데이터로더 설정 완료, 배치는 훈련 중 GPU로 자동 이동됨
2025-06-23 02:13:17,385 - INFO - 고급 모델 초기화 및 훈련 준비 완료



⚙️ 고급 훈련 설정:
   🔸 모델 입력 차원: 511
   🔸 잠재 벡터 차원: 128
   🔸 Beta 값: 4.0
   🔸 기본 학습률: 0.0002
   🔸 가중치 감쇠: 1e-05
   🔸 Dropout 비율: 0.15
   🔸 배치 크기: 512
   🔸 에포크 수: 100
   🔸 Warm-up 에포크: 10
   🔸 Early Stopping 인내심: 15
   🔸 Gradient Clipping: 1.0
   🔸 Mixed Precision: ✅
   🔸 훈련 배치 수: 2,279
   🔸 검증 배치 수: 403

💾 체크포인트 설정:
   🔸 체크포인트 디렉토리: notebooks/advanced_b_vae_checkpoints
   🔸 실험 ID: 20250623_021157
   🔸 최적 모델 저장: ✅
   🔸 주기적 저장: 매 10 에포크
   🔸 최적 모델 경로: notebooks/advanced_b_vae_checkpoints\best_model_20250623_021157.pth
   🔸 최신 모델 경로: notebooks/advanced_b_vae_checkpoints\latest_model_20250623_021157.pth
   🔸 설정 파일 경로: notebooks/advanced_b_vae_checkpoints\config_20250623_021157.json


In [14]:
# 고급 훈련 루프
print("\n📊 6단계: 고급 모델 훈련")
print("-" * 40)

logger.info("고급 Beta-VAE 훈련 시작")
print("🚀 고급 Beta-VAE 훈련 시작...")
print("   🔥 GPU 가속 | Mixed Precision | Early Stopping | LR Scheduling")

# 훈련 기록을 위한 리스트
train_losses, val_losses = [], []
recon_losses, kld_losses = [], []
learning_rates = []
best_val_loss = float('inf')

# 검증 함수 정의
def validate_model(model, val_loader, epoch):
    model.eval()
    val_total_loss = 0
    val_recon_loss = 0
    val_kld_loss = 0
    
    # Beta weight 계산 (warm-up 지원)
    if epoch < warmup_epochs:
        beta_weight = (epoch + 1) / warmup_epochs
    else:
        beta_weight = 1.0
    
    with torch.no_grad():
        for x_batch, m_batch in val_loader:
            # 배치를 GPU로 이동
            x_batch = x_batch.to(device, non_blocking=True)
            m_batch = m_batch.to(device, non_blocking=True)
            
            if scaler:  # Mixed precision
                with autocast():
                    recon_batch, mu, logvar = model(x_batch)
                    total_loss, recon_loss, kld_loss = model.loss_function(
                        recon_batch, x_batch, mu, logvar, m_batch, beta_weight
                    )
            else:
                recon_batch, mu, logvar = model(x_batch)
                total_loss, recon_loss, kld_loss = model.loss_function(
                    recon_batch, x_batch, mu, logvar, m_batch, beta_weight
                )
            
            val_total_loss += total_loss.item()
            val_recon_loss += recon_loss.item()
            val_kld_loss += kld_loss.item()
    
    return (val_total_loss / len(val_loader), 
            val_recon_loss / len(val_loader), 
            val_kld_loss / len(val_loader))

start_time = datetime.now()

for epoch in range(epochs):
    # 훈련 단계
    vae.train()
    epoch_total_loss = 0
    epoch_recon_loss = 0
    epoch_kld_loss = 0
    
    # Learning Rate Warm-up
    if epoch < warmup_epochs:
        current_lr = warmup_scheduler.step(epoch)
        beta_weight = (epoch + 1) / warmup_epochs  # Beta annealing
    else:
        current_lr = optimizer.param_groups[0]['lr']
        beta_weight = 1.0
    
    learning_rates.append(current_lr)
    
    # 에포크별 진행 바
    progress_bar = tqdm(train_dataloader, desc=f"   에포크 {epoch+1:3d}/{epochs}", leave=False)
    
    for batch_idx, (x_batch, m_batch) in enumerate(progress_bar):
        # 배치를 GPU로 이동
        x_batch = x_batch.to(device, non_blocking=True)
        m_batch = m_batch.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if scaler:  # Mixed Precision Training
            with autocast():
                recon_batch, mu, logvar = vae(x_batch)
                total_loss, recon_loss, kld_loss = vae.loss_function(
                    recon_batch, x_batch, mu, logvar, m_batch, beta_weight
                )
            
            scaler.scale(total_loss).backward()
            
            # Gradient Clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(vae.parameters(), grad_clip_value)
            
            scaler.step(optimizer)
            scaler.update()
        else:
            recon_batch, mu, logvar = vae(x_batch)
            total_loss, recon_loss, kld_loss = vae.loss_function(
                recon_batch, x_batch, mu, logvar, m_batch, beta_weight
            )
            
            total_loss.backward()
            
            # Gradient Clipping
            torch.nn.utils.clip_grad_norm_(vae.parameters(), grad_clip_value)
            
            optimizer.step()
        
        # 손실 누적
        epoch_total_loss += total_loss.item()
        epoch_recon_loss += recon_loss.item()
        epoch_kld_loss += kld_loss.item()
        
        # 진행 바 업데이트
        progress_bar.set_postfix({
            'Loss': f'{total_loss.item():.4f}',
            'Recon': f'{recon_loss.item():.4f}',
            'KLD': f'{kld_loss.item():.4f}',
            'LR': f'{current_lr:.2e}',
            'β': f'{beta_weight:.2f}'
        })
    
    # 에포크별 평균 손실 계산
    avg_train_loss = epoch_total_loss / len(train_dataloader)
    avg_recon_loss = epoch_recon_loss / len(train_dataloader)
    avg_kld_loss = epoch_kld_loss / len(train_dataloader)
    
    # 검증 단계
    val_loss, val_recon_loss, val_kld_loss = validate_model(vae, val_dataloader, epoch)
    
    # 손실 기록
    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)
    recon_losses.append(avg_recon_loss)
    kld_losses.append(avg_kld_loss)
    
    # Learning Rate Scheduling
    if epoch >= warmup_epochs:
        cosine_scheduler.step()
    plateau_scheduler.step(val_loss)
    
    # 주기적 출력 (5에포크마다)
    if (epoch + 1) % 5 == 0:
        elapsed_time = datetime.now() - start_time
        print(f"   📈 에포크 {epoch+1:3d}/{epochs} | "
              f"훈련: {avg_train_loss:.4f} | "
              f"검증: {val_loss:.4f} | "
              f"재구성: {avg_recon_loss:.4f} | "
              f"KLD: {avg_kld_loss:.4f} | "
              f"LR: {current_lr:.2e} | "
              f"시간: {elapsed_time}")
        logger.info(f"에포크 {epoch+1} - 훈련 손실: {avg_train_loss:.4f}, 검증 손실: {val_loss:.4f}")
    
    # 체크포인트 저장 (최적 모델 및 주기적 저장)
    is_best = checkpoint_manager.update_best(val_loss, epoch)
    if is_best:
        best_val_loss = val_loss
        print(f"   🎯 새로운 최적 모델! 검증 손실: {val_loss:.4f}")
    
    # 체크포인트 저장
    additional_info = {
        'learning_rates': learning_rates,
        'recon_losses': recon_losses,
        'kld_losses': kld_losses,
        'beta_weight': beta_weight,
        'current_lr': current_lr
    }
    
    checkpoint_manager.save_checkpoint(
        epoch=epoch,
        model=vae,
        optimizer=optimizer,
        train_losses=train_losses,
        val_losses=val_losses,
        hyperparameters=hyperparameters,
        is_best=is_best,
        additional_info=additional_info
    )
    
    # Early Stopping 확인
    if early_stopping(val_loss, vae, epoch):
        print(f"\n🛑 Early Stopping 발동! (에포크 {epoch+1})")
        print(f"   🔸 최적 검증 손실: {early_stopping.best_loss:.4f}")
        print(f"   🔸 인내심 카운터: {early_stopping.counter}/{patience}")
        print(f"   🔸 최적 모델이 복원되었습니다")
        logger.info(f"Early stopping at epoch {epoch+1}")
        
        # 최종 체크포인트 저장 (Early stopping 정보 포함)
        final_info = additional_info.copy()
        final_info.update({
            'early_stopped': True,
            'stopped_epoch': early_stopping.stopped_epoch,
            'final_patience_counter': early_stopping.counter
        })
        
        checkpoint_manager.save_checkpoint(
            epoch=epoch,
            model=vae,
            optimizer=optimizer,
            train_losses=train_losses,
            val_losses=val_losses,
            hyperparameters=hyperparameters,
            is_best=False,  # 이미 최적 모델로 복원됨
            additional_info=final_info
        )
        break

total_training_time = datetime.now() - start_time
final_epoch = epoch + 1

print(f"\n✅ 고급 훈련 완료!")
print(f"   🔸 총 훈련 시간: {total_training_time}")
print(f"   🔸 실제 훈련 에포크: {final_epoch}/{epochs}")
print(f"   🔸 최종 훈련 손실: {train_losses[-1]:.4f}")
print(f"   🔸 최종 검증 손실: {val_losses[-1]:.4f}")
print(f"   🔸 최적 검증 손실: {best_val_loss:.4f}")
print(f"   🔸 최종 재구성 손실: {recon_losses[-1]:.4f}")
print(f"   🔸 최종 KLD 손실: {kld_losses[-1]:.4f}")
print(f"   🔸 최종 학습률: {learning_rates[-1]:.2e}")

logger.info(f"고급 훈련 완료 - 총 시간: {total_training_time}, 최종 검증 손실: {val_losses[-1]:.4f}")


2025-06-23 02:13:17,424 - INFO - 고급 Beta-VAE 훈련 시작



📊 6단계: 고급 모델 훈련
----------------------------------------
🚀 고급 Beta-VAE 훈련 시작...
   🔥 GPU 가속 | Mixed Precision | Early Stopping | LR Scheduling


   에포크   1/100:   0%|          | 0/2279 [00:00<?, ?it/s]C:\Users\Administrator\AppData\Local\Temp\ipykernel_5592\1773627523.py:84: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5592\1773627523.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
2025-06-23 02:14:24,947 - INFO - 💾 최적 모델 저장됨: notebooks/advanced_b_vae_checkpoints\best_model_20250623_021157.pth


   🎯 새로운 최적 모델! 검증 손실: 1.1297


2025-06-23 02:15:39,730 - INFO - 💾 최적 모델 저장됨: notebooks/advanced_b_vae_checkpoints\best_model_20250623_021157.pth                


...

   📈 에포크  10/100 | 훈련: 1.0353 | 검증: 1.0394 | 재구성: 1.0210 | KLD: 0.0036 | LR: 2.00e-04 | 시간: 0:11:56.271267


2025-06-23 02:31:08,725 - INFO - 에포크 15 - 훈련 손실: 1.0420, 검증 손실: 1.0446                                                            


   📈 에포크  15/100 | 훈련: 1.0420 | 검증: 1.0446 | 재구성: 1.0210 | KLD: 0.0052 | LR: 9.95e-05 | 시간: 0:17:51.298756


2025-06-23 02:35:53,651 - INFO - 🔄 최적 가중치로 복원됨 (에포크 3)                                                                           
2025-06-23 02:35:53,651 - INFO - Early stopping at epoch 19
2025-06-23 02:35:53,671 - INFO - 고급 훈련 완료 - 총 시간: 0:22:36.245489, 최종 검증 손실: 1.0619



🛑 Early Stopping 발동! (에포크 19)
   🔸 최적 검증 손실: 1.0215
   🔸 인내심 카운터: 15/15
   🔸 최적 모델이 복원되었습니다

✅ 고급 훈련 완료!
   🔸 총 훈련 시간: 0:22:36.245489
   🔸 실제 훈련 에포크: 19/100
   🔸 최종 훈련 손실: 1.0549
   🔸 최종 검증 손실: 1.0619
   🔸 최적 검증 손실: 1.0215
   🔸 최종 재구성 손실: 1.0210
   🔸 최종 KLD 손실: 0.0085
   🔸 최종 학습률: 9.81e-05


In [15]:
# 고급 임베딩 추출
print("\n📊 7단계: 최적화된 잠재 임베딩 추출")
print("-" * 40)

logger.info("최적화된 잠재 임베딩 추출 시작")

def extract_latent_advanced(model, X, batch_size=2000):
    """GPU 메모리 효율적인 배치 임베딩 추출"""
    model.eval()
    embeddings_list = []
    
    # GPU 메모리에 맞춰 배치 크기 조정
    if device.type == 'cuda':
        available_memory = torch.cuda.get_device_properties(0).total_memory
        current_memory = torch.cuda.memory_allocated()
        free_memory = available_memory - current_memory
        # 안전 마진 고려하여 배치 크기 조정
        if free_memory < 2 * 1024**3:  # 2GB 미만
            batch_size = 1000
        elif free_memory < 4 * 1024**3:  # 4GB 미만
            batch_size = 1500
    
    print(f"🧠 최적화된 배치 크기로 잠재 벡터 추출 중... (배치: {batch_size})")
    
    with torch.no_grad():
        for i in tqdm(range(0, len(X), batch_size), desc="   임베딩 추출"):
            batch = X[i:i+batch_size].to(device, non_blocking=True)
            
            if scaler:  # Mixed precision 지원
                with autocast():
                    mu, _ = model.encode(batch)
            else:
                mu, _ = model.encode(batch)
            
            embeddings_list.append(mu.cpu())
    
    return torch.cat(embeddings_list, dim=0)

# 전체 데이터에서 임베딩 추출
embedding_tensor = extract_latent_advanced(vae, X)
embeddings = embedding_tensor.numpy()

# 훈련/검증 데이터별 임베딩도 추출
print("🔄 훈련/검증 데이터별 임베딩 추출 중...")
train_embeddings = extract_latent_advanced(vae, X[train_indices]).numpy()
val_embeddings = extract_latent_advanced(vae, X[val_indices]).numpy()

print(f"\n📊 추출된 임베딩 정보:")
print(f"   🔸 전체 임베딩 형태: {embeddings.shape}")
print(f"   🔸 훈련 임베딩 형태: {train_embeddings.shape}")
print(f"   🔸 검증 임베딩 형태: {val_embeddings.shape}")
print(f"   🔸 임베딩 차원: {embeddings.shape[1]}")
print(f"   🔸 총 데이터 수: {embeddings.shape[0]:,}")
print(f"   🔸 메모리 사용량: {embeddings.nbytes / 1024**2:.2f} MB")

# 임베딩 품질 분석
print(f"\n📈 임베딩 품질 분석:")
mean_embedding = np.mean(embeddings, axis=0)
std_embedding = np.std(embeddings, axis=0)
train_mean = np.mean(train_embeddings, axis=0)
val_mean = np.mean(val_embeddings, axis=0)

print(f"   🔸 전체 평균 범위: [{np.min(mean_embedding):.4f}, {np.max(mean_embedding):.4f}]")
print(f"   🔸 전체 표준편차 범위: [{np.min(std_embedding):.4f}, {np.max(std_embedding):.4f}]")
print(f"   🔸 훈련/검증 평균 차이: {np.mean(np.abs(train_mean - val_mean)):.6f}")

# 차원 활용도 분석 (0에 가까운 차원 체크)
inactive_dims = np.sum(np.abs(mean_embedding) < 0.01)
print(f"   🔸 비활성 차원 수: {inactive_dims}/{latent_dim} ({inactive_dims/latent_dim*100:.1f}%)")

# 훈련 안정성 분석
if len(train_losses) > 10:
    final_10_train = train_losses[-10:]
    final_10_val = val_losses[-10:]
    train_stability = np.std(final_10_train) / np.mean(final_10_train)
    val_stability = np.std(final_10_val) / np.mean(final_10_val)
    print(f"   🔸 훈련 안정성 (CV): {train_stability:.4f}")
    print(f"   🔸 검증 안정성 (CV): {val_stability:.4f}")

logger.info(f"최적화된 임베딩 추출 완료: {embeddings.shape}")

# GPU 메모리 정리
if device.type == 'cuda':
    torch.cuda.empty_cache()
    print(f"   🔧 GPU 메모리 정리 완료")


2025-06-23 02:35:53,696 - INFO - 최적화된 잠재 임베딩 추출 시작



📊 7단계: 최적화된 잠재 임베딩 추출
----------------------------------------
🧠 최적화된 배치 크기로 잠재 벡터 추출 중... (배치: 2000)


   임베딩 추출:   0%|          | 0/687 [00:00<?, ?it/s]C:\Users\Administrator\AppData\Local\Temp\ipykernel_5592\765444376.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
   임베딩 추출: 100%|██████████| 687/687 [00:04<00:00, 163.26it/s]


🔄 훈련/검증 데이터별 임베딩 추출 중...
🧠 최적화된 배치 크기로 잠재 벡터 추출 중... (배치: 2000)


   임베딩 추출: 100%|██████████| 584/584 [00:03<00:00, 169.96it/s]


🧠 최적화된 배치 크기로 잠재 벡터 추출 중... (배치: 2000)


   임베딩 추출: 100%|██████████| 103/103 [00:00<00:00, 166.41it/s]



📊 추출된 임베딩 정보:
   🔸 전체 임베딩 형태: (1372280, 128)
   🔸 훈련 임베딩 형태: (1166438, 128)
   🔸 검증 임베딩 형태: (205842, 128)
   🔸 임베딩 차원: 128
   🔸 총 데이터 수: 1,372,280
   🔸 메모리 사용량: 335.03 MB

📈 임베딩 품질 분석:


2025-06-23 02:36:12,304 - INFO - 최적화된 임베딩 추출 완료: (1372280, 128)


   🔸 전체 평균 범위: [-0.0005, 0.0005]
   🔸 전체 표준편차 범위: [0.0000, 0.0000]
   🔸 훈련/검증 평균 차이: 0.000000
   🔸 비활성 차원 수: 128/128 (100.0%)
   🔸 훈련 안정성 (CV): 0.0054
   🔸 검증 안정성 (CV): 0.0065
   🔧 GPU 메모리 정리 완료


In [16]:
# 고급 마무리 및 성능 분석
print("\n📊 8단계: 고급 마무리 및 성능 분석")
print("-" * 40)

# 훈련 성능 시각화 및 분석
print("📈 훈련 성능 분석:")
if len(train_losses) > 0:
    # 손실 수렴 분석
    improvement_rate = (train_losses[0] - train_losses[-1]) / train_losses[0] * 100
    val_improvement_rate = (val_losses[0] - val_losses[-1]) / val_losses[0] * 100
    
    print(f"   🔸 훈련 손실 개선률: {improvement_rate:.2f}%")
    print(f"   🔸 검증 손실 개선률: {val_improvement_rate:.2f}%")
    
    # 오버피팅 감지
    final_gap = abs(train_losses[-1] - val_losses[-1])
    overfitting_ratio = final_gap / val_losses[-1]
    print(f"   🔸 최종 훈련/검증 손실 차이: {final_gap:.4f}")
    print(f"   🔸 오버피팅 비율: {overfitting_ratio:.4f}")
    
    if overfitting_ratio > 0.1:
        print("   ⚠️ 경고: 오버피팅 가능성 감지 (10% 이상 차이)")
    else:
        print("   ✅ 양호: 오버피팅 수준이 적절함")

# 최종 체크포인트 저장 및 정리
print(f"\n💾 최종 체크포인트 저장:")

# 최종 추가 정보 수집
final_additional_info = {
    'learning_rates': learning_rates,
    'recon_losses': recon_losses,
    'kld_losses': kld_losses,
    'total_training_time': str(total_training_time),
    'final_epoch': final_epoch,
    'training_completed': True,
    'compression_ratio': latent_dim / max_dim,
    'model_size_mb': total_params * 4 / 1024**2
}

# 최종 체크포인트 저장
final_checkpoint = checkpoint_manager.save_checkpoint(
    epoch=final_epoch-1,
    model=vae,
    optimizer=optimizer,
    train_losses=train_losses,
    val_losses=val_losses,
    hyperparameters=hyperparameters,
    is_best=(val_losses[-1] == best_val_loss),
    additional_info=final_additional_info
)

print(f"   ✅ 최종 체크포인트 저장 완료")
print(f"   📁 최적 모델 경로: {checkpoint_manager.best_checkpoint_path}")
print(f"   📁 최신 모델 경로: {checkpoint_manager.latest_checkpoint_path}")
print(f"   📁 설정 파일 경로: {checkpoint_manager.config_path}")

# 체크포인트 디렉토리 내용 확인
checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pth') or f.endswith('.json')]
print(f"\n📂 저장된 체크포인트 파일들:")
for file in sorted(checkpoint_files):
    file_path = os.path.join(checkpoint_dir, file)
    file_size = os.path.getsize(file_path) / 1024**2  # MB
    print(f"   📄 {file} ({file_size:.2f} MB)")

# 체크포인트 로딩 예시 함수 생성
def load_checkpoint_example():
    """체크포인트 로딩 예시 코드"""
    example_code = f'''
# 최적 모델 로딩 예시:
checkpoint = torch.load('{checkpoint_manager.best_checkpoint_path}')
model = AdvancedBetaVAE(
    input_dim={input_dim}, 
    latent_dim={latent_dim}, 
    beta={beta},
    dropout_rate={dropout_rate}
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

# 훈련 재개 예시:
optimizer = optim.AdamW(model.parameters(), lr={base_lr}, weight_decay={weight_decay})
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1
train_losses = checkpoint['train_losses']
val_losses = checkpoint['val_losses']

print(f"체크포인트 로딩 완료:")
print(f"  에포크: {{checkpoint['epoch']}}")
print(f"  최적 검증 손실: {{checkpoint['best_val_loss']:.4f}}")
print(f"  실험 ID: {{checkpoint['experiment_id']}}")
'''
    return example_code

print(f"\n📝 체크포인트 로딩 가이드:")
print("   🔸 최적 모델 사용 시: best_model_*.pth 파일 로딩")
print("   🔸 훈련 재개 시: latest_model_*.pth 파일 로딩")
print("   🔸 설정 확인 시: config_*.json 파일 참조")
print(f"   🔸 주기적 체크포인트: epoch_XXX_{experiment_id}.pth 파일들")

print("\n📋 데이터베이스 저장 예시 코드:")
print("""
# pgvector 확장 활성화 (필요시)
cursor.execute('CREATE EXTENSION IF NOT EXISTS vector;')

# 고급 임베딩 테이블 생성 (pgvector 사용)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS advanced_beta_vae_embeddings (
        id SERIAL PRIMARY KEY,
        original_vector_id INTEGER,
        vector_source VARCHAR(20),  -- 'wavelet' or 'dct'
        embedding_type VARCHAR(30) DEFAULT 'advanced_beta_vae_128d',
        vae_embedding vector(128),  -- pgvector 타입 사용
        training_metadata JSONB,   -- 훈련 정보 저장
        quality_metrics JSONB,     -- 품질 지표 저장
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE(original_vector_id, embedding_type)
    );
    
    -- 벡터 검색 성능을 위한 인덱스 생성
    CREATE INDEX IF NOT EXISTS idx_vae_embedding_cosine 
    ON advanced_beta_vae_embeddings USING ivfflat (vae_embedding vector_cosine_ops);
    
    CREATE INDEX IF NOT EXISTS idx_vae_embedding_l2 
    ON advanced_beta_vae_embeddings USING ivfflat (vae_embedding vector_l2_ops);
''')

# 메타데이터와 함께 임베딩 저장
metadata = {
    'final_epoch': final_epoch,
    'best_val_loss': best_val_loss,
    'train_loss': train_losses[-1],
    'val_loss': val_losses[-1],
    'model_params': total_params,
    'compression_ratio': latent_dim/max_dim
}

for i, embedding in enumerate(embeddings):
    # pgvector는 리스트를 직접 받음 (JSON 변환 불필요)
    embedding_list = embedding.tolist()
    cursor.execute(
        \"\"\"INSERT INTO advanced_beta_vae_embeddings 
           (original_vector_id, embedding_type, vae_embedding, training_metadata) 
           VALUES (%s, %s, %s, %s) ON CONFLICT (original_vector_id, embedding_type) 
           DO UPDATE SET vae_embedding = EXCLUDED.vae_embedding\"\"\",
        (i, 'advanced_beta_vae_128d', embedding_list, json.dumps(metadata))
    )
conn.commit()

# 벡터 유사도 검색 예시
query_embedding = embeddings[0].tolist()  # 첫 번째 임베딩을 쿼리로 사용
cursor.execute('''
    SELECT original_vector_id, vector_source, 
           vae_embedding <=> %s AS cosine_distance,
           1 - (vae_embedding <=> %s) AS cosine_similarity
    FROM advanced_beta_vae_embeddings 
    ORDER BY vae_embedding <=> %s 
    LIMIT 10;
''', (query_embedding, query_embedding, query_embedding))

similar_vectors = cursor.fetchall()
print("\\n🔍 유사도 검색 결과 (상위 10개):")
for row in similar_vectors:
    vector_id, source, distance, similarity = row
    print(f"   벡터 ID: {vector_id}, 소스: {source}, 유사도: {similarity:.4f}")
""")

# 최종 종합 요약
print(f"\n🎉 고급 Beta-VAE 파이프라인 완료!")
print("=" * 70)
print(f"📊 종합 성능 요약:")
print(f"   🔸 입력 벡터 수: {len(all_vectors):,}개")
print(f"   🔸 원본 최대 차원: {max_dim}")
print(f"   🔸 압축된 임베딩 차원: {latent_dim}")
print(f"   🔸 압축률: {(latent_dim/max_dim)*100:.1f}%")
print(f"   🔸 모델 파라미터 수: {total_params:,}")
print(f"   🔸 실제 훈련 에포크: {final_epoch}/{epochs}")
print(f"   🔸 최종 훈련 손실: {train_losses[-1]:.4f}")
print(f"   🔸 최종 검증 손실: {val_losses[-1]:.4f}")
print(f"   🔸 최적 검증 손실: {best_val_loss:.4f}")
print(f"   🔸 총 훈련 시간: {total_training_time}")

print(f"\n🚀 적용된 고급 기법:")
print(f"   ✅ GPU 가속 훈련 ({device})")
print(f"   ✅ Mixed Precision Training ({'활성' if scaler else '비활성'})")
print(f"   ✅ Early Stopping (patience={patience})")
print(f"   ✅ Learning Rate Warm-up ({warmup_epochs} 에포크)")
print(f"   ✅ Cosine Annealing LR Scheduler")
print(f"   ✅ Gradient Clipping (max_norm={grad_clip_value})")
print(f"   ✅ Layer Normalization + Dropout ({dropout_rate})")
print(f"   ✅ Weight Decay ({weight_decay})")
print(f"   ✅ Xavier 가중치 초기화")
print(f"   ✅ 훈련/검증 데이터 분할 (85%/15%)")

logger.info("고급 Beta-VAE 파이프라인 성공적으로 완료")

# 리소스 정리
print("\n🔧 리소스 정리 중...")
cursor.close()
conn.close()
logger.info("데이터베이스 연결 종료")

# GPU 메모리 정리
if device.type == 'cuda':
    torch.cuda.empty_cache()
    print("🎯 GPU 메모리 정리 완료")

print("✅ 모든 리소스가 정리되었습니다.")

print("\n🎯 고급 임베딩 활용 방안:")
print("   📈 유사도 검색: 코사인/유클리드 거리 기반 벡터 검색")
print("   🔍 클러스터링: K-means, DBSCAN 등으로 패턴 분석")
print("   🎯 분류/회귀: 다운스트림 태스크의 특징으로 활용")
print("   📊 차원 축소 시각화: t-SNE, UMAP으로 고차원 데이터 시각화")
print("   🔄 전이 학습: 사전 훈련된 특징으로 새로운 태스크 적용")
print("   💾 모델 체크포인트: 재훈련 없이 임베딩 생성 재개 가능")

print(f"\n💡 성능 최적화 제안:")
if overfitting_ratio > 0.1:
    print("   📝 Dropout 비율 증가 또는 더 강한 정규화 고려")
if inactive_dims > latent_dim * 0.1:
    print("   📝 잠재 차원 축소 또는 β값 조정 고려")
if final_epoch == epochs:
    print("   📝 더 많은 에포크 또는 낮은 학습률로 추가 훈련 고려")

print(f"\n💾 체크포인트 관리 및 백업 가이드:")
print("   📁 체크포인트 구조:")
print(f"     • best_model_{experiment_id}.pth: 최적 성능 모델")
print(f"     • latest_model_{experiment_id}.pth: 최신 훈련 상태")
print(f"     • config_{experiment_id}.json: 실험 설정 및 메타데이터")
print(f"     • epoch_XXX_{experiment_id}.pth: 주기적 백업 (매 {save_every_n_epochs} 에포크)")
print("   🔄 체크포인트 복원 시나리오:")
print("     • 최적 모델 배포: best_model_*.pth 로딩")
print("     • 훈련 중단 후 재개: latest_model_*.pth 로딩")
print("     • 특정 시점 분석: epoch_XXX_*.pth 로딩")
print("     • 실험 재현: config_*.json 파일 참조")
print("   📝 관리 권장사항:")
print("     • 정기적 백업: 중요한 체크포인트를 별도 위치에 백업")
print("     • 저장공간 관리: 오래된 주기적 체크포인트 정리")
print("     • 메타데이터 활용: config.json으로 실험 추적 및 비교")
print("     • 버전 관리: Git LFS로 체크포인트 버전 관리")

print(f"\n🚀 향상된 최적화 기능:")
print(f"   ✅ 고급 체크포인트 관리 시스템")
print(f"   ✅ 최적 모델 자동 저장 ({checkpoint_manager.best_checkpoint_path})")
print(f"   ✅ 향상된 Early Stopping (patience={patience}, 최적 가중치 복원)")
print(f"   ✅ 주기적 체크포인트 저장 (매 {save_every_n_epochs} 에포크)")
print(f"   ✅ 포괄적 메타데이터 저장 (훈련 기록, 설정, 성능 지표)")
print(f"   ✅ 실험 ID 기반 파일 관리 ({experiment_id})")
print(f"   ✅ 체크포인트 로딩 가이드 제공")

print("\n" + "=" * 70)
print("🎊 고급 Beta-VAE 임베딩 시스템이 성공적으로 완료되었습니다!")
print("=" * 70)


2025-06-23 02:36:12,470 - INFO - 고급 Beta-VAE 파이프라인 성공적으로 완료



📊 8단계: 고급 마무리 및 성능 분석
----------------------------------------
📈 훈련 성능 분석:
   🔸 훈련 손실 개선률: 79.90%
   🔸 검증 손실 개선률: 6.00%
   🔸 최종 훈련/검증 손실 차이: 0.0069
   🔸 오버피팅 비율: 0.0065
   ✅ 양호: 오버피팅 수준이 적절함

💾 최종 체크포인트 저장:
   ✅ 최종 체크포인트 저장 완료
   📁 최적 모델 경로: notebooks/advanced_b_vae_checkpoints\best_model_20250623_021157.pth
   📁 최신 모델 경로: notebooks/advanced_b_vae_checkpoints\latest_model_20250623_021157.pth
   📁 설정 파일 경로: notebooks/advanced_b_vae_checkpoints\config_20250623_021157.json

📂 저장된 체크포인트 파일들:
   📄 best_model_20250623_021157.pth (10.20 MB)
   📄 config_20250623_020449.json (0.00 MB)
   📄 config_20250623_021157.json (0.00 MB)
   📄 epoch_010_20250623_021157.pth (10.20 MB)
   📄 latest_model_20250623_021157.pth (10.20 MB)

📝 체크포인트 로딩 가이드:
   🔸 최적 모델 사용 시: best_model_*.pth 파일 로딩
   🔸 훈련 재개 시: latest_model_*.pth 파일 로딩
   🔸 설정 확인 시: config_*.json 파일 참조
   🔸 주기적 체크포인트: epoch_XXX_20250623_021157.pth 파일들

📋 데이터베이스 저장 예시 코드:

# pgvector 확장 활성화 (필요시)
cursor.execute('CREATE EXTENSION IF NOT EXISTS vector

2025-06-23 02:36:12,630 - INFO - 데이터베이스 연결 종료


🎯 GPU 메모리 정리 완료
✅ 모든 리소스가 정리되었습니다.

🎯 고급 임베딩 활용 방안:
   📈 유사도 검색: 코사인/유클리드 거리 기반 벡터 검색
   🔍 클러스터링: K-means, DBSCAN 등으로 패턴 분석
   🎯 분류/회귀: 다운스트림 태스크의 특징으로 활용
   📊 차원 축소 시각화: t-SNE, UMAP으로 고차원 데이터 시각화
   🔄 전이 학습: 사전 훈련된 특징으로 새로운 태스크 적용
   💾 모델 체크포인트: 재훈련 없이 임베딩 생성 재개 가능

💡 성능 최적화 제안:
   📝 잠재 차원 축소 또는 β값 조정 고려

💾 체크포인트 관리 및 백업 가이드:
   📁 체크포인트 구조:
     • best_model_20250623_021157.pth: 최적 성능 모델
     • latest_model_20250623_021157.pth: 최신 훈련 상태
     • config_20250623_021157.json: 실험 설정 및 메타데이터
     • epoch_XXX_20250623_021157.pth: 주기적 백업 (매 10 에포크)
   🔄 체크포인트 복원 시나리오:
     • 최적 모델 배포: best_model_*.pth 로딩
     • 훈련 중단 후 재개: latest_model_*.pth 로딩
     • 특정 시점 분석: epoch_XXX_*.pth 로딩
     • 실험 재현: config_*.json 파일 참조
   📝 관리 권장사항:
     • 정기적 백업: 중요한 체크포인트를 별도 위치에 백업
     • 저장공간 관리: 오래된 주기적 체크포인트 정리
     • 메타데이터 활용: config.json으로 실험 추적 및 비교
     • 버전 관리: Git LFS로 체크포인트 버전 관리

🚀 향상된 최적화 기능:
   ✅ 고급 체크포인트 관리 시스템
   ✅ 최적 모델 자동 저장 (notebooks/advanced_b_vae_checkpoints\best_model_20250623_021157.pth)
 

In [17]:
# 체크포인트 로딩 및 사용 예시
print("📋 체크포인트 로딩 및 사용 예시:")
print("=" * 50)

# 체크포인트 로딩 함수 정의
def load_best_checkpoint(checkpoint_path):
    """최적 체크포인트 로딩 함수"""
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        print(f"✅ 체크포인트 로딩 성공: {checkpoint_path}")
        print(f"   🔸 실험 ID: {checkpoint.get('experiment_id', 'N/A')}")
        print(f"   🔸 에포크: {checkpoint.get('epoch', 'N/A')}")
        print(f"   🔸 최적 검증 손실: {checkpoint.get('best_val_loss', 'N/A'):.4f}")
        print(f"   🔸 훈련 완료 여부: {checkpoint.get('training_completed', False)}")
        print(f"   🔸 조기 종료 여부: {checkpoint.get('early_stopped', False)}")
        
        # 모델 재생성 및 가중치 로딩
        model = AdvancedBetaVAE(
            input_dim=checkpoint['hyperparameters']['input_dim'],
            latent_dim=checkpoint['hyperparameters']['latent_dim'],
            beta=checkpoint['hyperparameters']['beta'],
            dropout_rate=checkpoint['hyperparameters']['dropout_rate']
        ).to(device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"   🧠 모델 재생성 및 가중치 로딩 완료")
        return model, checkpoint
        
    except Exception as e:
        print(f"❌ 체크포인트 로딩 실패: {e}")
        return None, None

def load_config(config_path):
    """설정 파일 로딩 함수"""
    try:
        with open(config_path, 'r') as f:
            config = json.load(f)
        print(f"⚙️ 설정 파일 로딩 성공: {config_path}")
        print(f"   🔸 모델 파라미터: {config.get('total_params', 'N/A'):,}")
        print(f"   🔸 압축률: {config.get('latent_dim', 0) / config.get('input_dim', 1) * 100:.1f}%")
        print(f"   🔸 사용 디바이스: {config.get('device', 'N/A')}")
        return config
    except Exception as e:
        print(f"❌ 설정 파일 로딩 실패: {e}")
        return None

print("\\n🔧 실제 사용 예시:")
print("# 최적 모델 로딩")
print(f"model, checkpoint = load_best_checkpoint('advanced_b_vae_checkpoints/best_model_20250623_021157.pth')")
print("\\n# 설정 확인")
print(f"config = load_config('advanced_b_vae_checkpoints/config_20250623_021157.json')")

print("\\n# 새로운 데이터에 대한 임베딩 생성")
print("# model.eval()")
print("# with torch.no_grad():")
print("#     new_embeddings = model.encode(new_data)[0]  # mu만 사용")

print("\\n# 훈련 재개 (latest 체크포인트 사용)")
print(f"# checkpoint = torch.load('advanced_b_vae_checkpoints/latest_model_20250623_021157.pth')")
print("# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])")
print("# start_epoch = checkpoint['epoch'] + 1")

logger.info("체크포인트 관리 시스템 예시 완료")


2025-06-23 02:36:12,650 - INFO - 체크포인트 관리 시스템 예시 완료


📋 체크포인트 로딩 및 사용 예시:
\n🔧 실제 사용 예시:
# 최적 모델 로딩
model, checkpoint = load_best_checkpoint('notebooks/advanced_b_vae_checkpoints\best_model_20250623_021157.pth')
\n# 설정 확인
config = load_config('notebooks/advanced_b_vae_checkpoints\config_20250623_021157.json')
\n# 새로운 데이터에 대한 임베딩 생성
# model.eval()
# with torch.no_grad():
#     new_embeddings = model.encode(new_data)[0]  # mu만 사용
\n# 훈련 재개 (latest 체크포인트 사용)
# checkpoint = torch.load('notebooks/advanced_b_vae_checkpoints\latest_model_20250623_021157.pth')
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# start_epoch = checkpoint['epoch'] + 1
